# XGBoost hyperparameter search

Goal: tune the weighted XGBoost with cross-validation on the **training set only**, scored on macro F1, and check whether tuning changes anything beyond the noise. The test set is used once, at the end.

In [ ]:
import sys, json, time
sys.path.append('..')

from pathlib import Path
import numpy as np
import pandas as pd
from scipy.stats import randint, uniform, loguniform
from sklearn.model_selection import (RandomizedSearchCV, RepeatedStratifiedKFold,
                                     StratifiedKFold, cross_val_score)
from src.config import SEED
from src.data import load_splits
from src.metrics import evaluate
from src.models import WeightedXGB

data, y_train, y_test = load_splits()
N_ITER = 30   # 30 candidates x 5 folds = 150 fits

## Choices

- **Features**: tuned on the `salary` set, the only one with a signal to learn. Without salary no model beats the baseline, so tuning would only fit noise.
- **Random search** over wide ranges instead of the 108-point grid of a typical study: a grid with maxima as its upper bounds can only report the edge (e.g. `max_depth=8` out of {4, 6, 8}) as the best value. We check below whether the best values sit on an edge.
- **Class weights computed inside each fold** (`WeightedXGB`): weights computed once on the whole training set would use the labels of the validation rows.
- **Scoring**: `f1_macro`, stratified 5-fold. Trees need no scaling, so the raw features are used.

In [2]:
param_distributions = {
    'n_estimators': randint(100, 501),
    'max_depth': randint(3, 11),
    'learning_rate': loguniform(0.02, 0.3),
    'subsample': uniform(0.6, 0.4),          # 0.6 to 1.0
    'colsample_bytree': uniform(0.6, 0.4),   # 0.6 to 1.0
}

search = RandomizedSearchCV(
    WeightedXGB(), param_distributions, n_iter=N_ITER, scoring='f1_macro',
    cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
    random_state=SEED, n_jobs=-1, refit=True,
)
t0 = time.time()
search.fit(data['salary']['X_train'], y_train)
print(f'{time.time() - t0:.0f} s')
print('best CV macro F1 :', round(search.best_score_, 4))
search.best_params_

374 s
best CV macro F1 : 0.9409


{'colsample_bytree': np.float64(0.6298202574719083),
 'learning_rate': np.float64(0.28953368345102004),
 'max_depth': 10,
 'n_estimators': 228,
 'subsample': np.float64(0.679486272613669)}

## Top candidates

Mean and standard deviation of the macro F1 over the 5 folds. If the best candidates are within one standard deviation of each other, their ranking is not meaningful.

In [3]:
cv = pd.DataFrame(search.cv_results_)
cols = ['mean_test_score', 'std_test_score', 'param_n_estimators', 'param_max_depth',
        'param_learning_rate', 'param_subsample', 'param_colsample_bytree']
cv.sort_values('rank_test_score')[cols].head(10).round(3).rename(columns=lambda c: c.replace('param_', ''))

,mean_test_score,std_test_score,n_estimators,max_depth,learning_rate,subsample,colsample_bytree
16,0.941,0.006,228,10,0.290,0.679,0.630
22,0.941,0.008,382,9,0.161,0.809,0.825
7,0.940,0.008,428,7,0.104,0.626,0.619
2,0.939,0.009,357,7,0.117,0.889,0.657
10,0.939,0.008,487,10,0.040,0.725,0.964
24,0.939,0.009,487,10,0.047,0.963,0.855
11,0.939,0.010,290,8,0.088,0.937,0.808
21,0.938,0.009,230,9,0.056,0.904,0.824
27,0.938,0.010,383,8,0.111,0.921,0.923
6,0.937,0.008,343,9,0.034,0.837,0.914


## Is the best value on the edge of the search range?

In [4]:
ranges = {'n_estimators': (100, 500), 'max_depth': (3, 10), 'learning_rate': (0.02, 0.3),
          'subsample': (0.6, 1.0), 'colsample_bytree': (0.6, 1.0)}
for name, (lo, hi) in ranges.items():
    v = search.best_params_[name]
    pos = (v - lo) / (hi - lo)
    print(f'{name:18s} best = {v:<10.4g} range [{lo}, {hi}]  position {pos:.2f}', '  <-- edge' if pos < 0.1 or pos > 0.9 else '')

n_estimators       best = 228        range [100, 500]  position 0.32 
max_depth          best = 10         range [3, 10]  position 1.00   <-- edge
learning_rate      best = 0.2895     range [0.02, 0.3]  position 0.96   <-- edge
subsample          best = 0.6795     range [0.6, 1.0]  position 0.20 
colsample_bytree   best = 0.6298     range [0.6, 1.0]  position 0.07   <-- edge


Repeated cross-validation, default vs tuned

Same folds for both models (5 folds × 3 repeats), on the training set. The paired difference shows whether the gain is larger than the fold-to-fold noise.

In [5]:
rcv = RepeatedStratifiedKFold(n_splits=5, n_repeats=3, random_state=SEED)
X_tr = data['salary']['X_train']

default_scores = cross_val_score(WeightedXGB(), X_tr, y_train, scoring='f1_macro', cv=rcv, n_jobs=-1)
tuned_scores = cross_val_score(WeightedXGB(**search.best_params_), X_tr, y_train, scoring='f1_macro', cv=rcv, n_jobs=-1)
diff = tuned_scores - default_scores

pd.DataFrame({
    'mean': [default_scores.mean(), tuned_scores.mean(), diff.mean()],
    'std': [default_scores.std(), tuned_scores.std(), diff.std()],
}, index=['default', 'tuned', 'tuned - default']).round(4)

,mean,std
default,0.9016,0.0085
tuned,0.9410,0.0057
tuned - default,0.0394,0.0071


## Test set (used once)

The tuned parameters are also applied to `no_salary` as a check: tuning cannot create a signal that is not there.

In [6]:
rows = []
for fs, d in data.items():
    for label, params in [('default', {}), ('tuned', search.best_params_)]:
        pred = WeightedXGB(**params).fit(d['X_train'], y_train).predict(d['X_test'])
        summary, table = evaluate(y_test, pred)
        rows.append({'features': fs, 'model': label, **summary,
                     'recall_1': table.loc[1, 'recall'], 'recall_2': table.loc[2, 'recall'],
                     'precision_1': table.loc[1, 'precision'], 'precision_2': table.loc[2, 'precision']})
pd.DataFrame(rows).set_index(['features', 'model']).round(3)

accuracy  macro_f1  recall_1  recall_2  precision_1  \
features  model                                                          
salary    default     0.979     0.909     0.800     0.865        0.681   
          tuned       0.987     0.926     0.638     0.850        0.927   
no_salary default     0.247     0.189     0.062     0.179        0.006   
          tuned       0.303     0.192     0.000     0.007        0.000   

                   precision_2  
features  model                 
salary    default        0.811  
          tuned          0.937  
no_salary default        0.060  
          tuned          0.051

In [7]:
Path('../results').mkdir(exist_ok=True)
best = {k: (v.item() if hasattr(v, 'item') else v) for k, v in search.best_params_.items()}
Path('../results/xgb_best_params.json').write_text(json.dumps(best, indent=2))
best

{'colsample_bytree': 0.6298202574719083,
 'learning_rate': 0.28953368345102004,
 'max_depth': 10,
 'n_estimators': 228,
 'subsample': 0.679486272613669}

## Notes

- **Search**: 30 random candidates × 5 stratified folds, scored on macro F1, on the `salary` set. Best CV macro F1: 0.941 with `n_estimators=228`, `max_depth=10`, `learning_rate=0.29`, `subsample=0.68`, `colsample_bytree=0.63`.
- **The top candidates are indistinguishable**: the ten best have a mean macro F1 between 0.937 and 0.941, a spread (0.004) smaller than the fold-to-fold standard deviation (0.006-0.010). Their ranking is not meaningful, and very different settings (depth 7 to 10, 228 to 487 trees) give the same score. The optimum is a plateau, not a point.
- **Three best values sit on the edge of their range** (`max_depth` 10 in [3, 10], `learning_rate` 0.29 in [0.02, 0.3], `colsample_bytree` 0.63 in [0.6, 1.0]). Since the plateau is flat, a wider range would probably change the score by less than the noise; the edge only says that the best region is "deep trees with a high learning rate".
- **Tuning does help here, beyond the noise.** In repeated stratified CV (5 folds × 3 repeats, same folds for both models) the macro F1 goes from 0.902 ± 0.009 (default) to 0.941 ± 0.006 (tuned). The paired difference is 0.039 with a standard deviation of 0.007 over the 15 folds: the gain is several times larger than the fold-to-fold variation.
- **On the test set** the gain is smaller: macro F1 0.909 → 0.926, accuracy 0.979 → 0.987. The tuned model trades recall of class 1 (0.800 → 0.638, i.e. 64 → 51 of the 80 rows) for precision (0.681 → 0.927, about 55 predictions of class 1). With 80 rows in class 1 (a recall interval of about ±0.1), this shift is within the noise of the test set, which is why the CV estimate (0.941) is more reliable than a single test score.
- **Tuning cannot create a signal**: without salary, the tuned parameters give a macro F1 of 0.192 (0.189 with default parameters), recall 0.000 on class 1, accuracy 0.303.
- **Reading**: even tuned, the model only exploits the salary, which is derived from the target (see commit 10 and the ablation: salary + job title gives a perfect score). Tuning improves how well the formula is recovered from salary alone, it does not make the model a better predictor of performance.
- **Limits**: a single test split with 80 rows of class 1, a search of 30 candidates, and tuning done only on the `salary` set. The best parameters are saved in `results/xgb_best_params.json` for the final comparison.